# Data Transformation

> Generated from [`04-Data-Transformation.py`](04-Data-Transformation.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Applying custom logic with `.apply()`/`.map()`, why vectorized operations should be the first reach instead of `.apply()`, sorting, distinct-value inspection, and value substitution with `.replace()`.

### Key points
- `Series.apply(func)` calls `func` per element; `DataFrame.apply(func, axis=0)` (default) calls it per column; `axis=1` calls it per row — use `axis=1` when a computation genuinely needs multiple columns together.
- `.map()` exists only on a Series — it substitutes values via a dict, another Series, or a function. Values missing from a dict/Series mapping become `NaN`.
- Vectorized operations (`df['a'] * df['b']`, comparisons, `.str`/`.dt` accessors, `np.where`) run as compiled loops and are dramatically faster than `.apply()`, which pays Python function-call overhead per row/element — `.apply()` is a last resort, not a first reach.
- `.sort_values(by=col_or_list, ascending=...)` sorts by column value(s) (accepts a list for multi-column sort); `.sort_index()` sorts by the row index label instead.
- `.value_counts()` gives distinct-value counts sorted descending by default (`normalize=True` gives proportions); `.unique()` gives distinct values in first-appearance order; `.nunique()` gives the count of distinct values.
- `.replace(old_to_new)` swaps specific values; unlike `.map()`, values not in the mapping are left **unchanged** rather than becoming `NaN`.

### Gotchas (the part worth revising twice)
- `.apply(axis=1)` looks convenient but was measured here at roughly 3000x slower than the vectorized equivalent on 200,000 rows — always check whether the same result can be expressed as direct arithmetic/comparison/built-in Series method first.
- `.map()` turns unmapped values into `NaN`; `.replace()` leaves them as-is — picking the wrong one silently loses or silently keeps data depending on which behavior you actually wanted.
- `.map()` is Series-only; calling it on a DataFrame is a common source of `AttributeError` — use `.apply()` or `.applymap()`-style per-column mapping for DataFrame-wide substitution instead.

### When it matters
Almost every real transformation — deriving a column from others, recoding categories, ranking, deduplicating value lists — goes through one of these tools; picking `.apply()` by default instead of vectorized code is the most common performance mistake in pandas scripts.

### One-line recap example
```python
import pandas as pd
s = pd.Series(["Widget", "Gadget"])
s.map({"Widget": "Hardware"})       # ['Hardware', nan]   - unmapped -> NaN
s.replace({"Widget": "Hardware"})   # ['Hardware', 'Gadget'] - unmapped unchanged
```

In [ ]:
# 01. .apply() - Row-wise vs Column-wise (axis=)
# 02. .map() - Series-only Element-wise Substitution/Lookup
# 03. Vectorized-Operation-First Principle (timing: vectorized vs .apply)
# 04. .sort_values() / .sort_index()
# 05. .value_counts(), .unique(), .nunique()
# 06. .replace()

import pandas as pd
import numpy as np
import time

sales = pd.DataFrame({
    "product": ["Widget", "Gadget", "Widget", "Gizmo", "Gadget", "Widget"],
    "region": ["East", "West", "East", "West", "East", "West"],
    "units": [120, 45, 80, 30, 60, 95],
    "unit_price": [9.99, 24.50, 9.99, 15.00, 24.50, 9.99],
})
print("sales:\n", sales, sep="")

## 01. .apply() - Row-wise vs Column-wise

- Series.apply(func)     -> calls func on each ELEMENT.
- DataFrame.apply(func, axis=0) (default) -> calls func on each COLUMN
  (a Series representing that column) - good for per-column reductions.
- DataFrame.apply(func, axis=1) -> calls func on each ROW (a Series
  representing that row) - good when a computation needs MULTIPLE
  columns together, which a single-column vectorized op can't express.

In [ ]:
print("sales['units'].apply(lambda x: x * 2) (Series.apply, element-wise):")
print(sales["units"].apply(lambda x: x * 2).tolist())

col_ranges = sales[["units", "unit_price"]].apply(lambda col: col.max() - col.min(), axis=0)
print("\nDataFrame.apply(axis=0) - per-COLUMN range (max-min) of units & unit_price:")
print(col_ranges)

row_totals = sales.apply(lambda row: row["units"] * row["unit_price"], axis=1)
print("\nDataFrame.apply(axis=1) - per-ROW total (units * unit_price, needs 2 columns together):")
print(row_totals.tolist())

## 02. .map() - Series-only Element-wise Substitution/Lookup

- .map() exists ONLY on a Series (not a DataFrame). It substitutes each
  value using a dict, another Series, or a function - a lookup/translation
  tool, distinct from .apply()'s more general "call this function" role.
- Values not found in a dict/Series mapping become NaN.

In [ ]:
category_lookup = {"Widget": "Hardware", "Gadget": "Electronics", "Gizmo": "Electronics"}
print("sales['product'].map(category_lookup) (dict lookup):")
print(sales["product"].map(category_lookup).tolist())

region_short = sales["region"].map(lambda r: r[0])
print("\nsales['region'].map(lambda r: r[0]) (function, first letter):", region_short.tolist())

unknown_lookup = {"Widget": "Hardware"}  # Gadget/Gizmo intentionally missing
print("\nsales['product'].map({'Widget': 'Hardware'}) (missing keys -> NaN):")
print(sales["product"].map(unknown_lookup).tolist())

## 03. Vectorized-Operation-First Principle

- Pandas/NumPy vectorized ops run in compiled C loops over contiguous
  memory. .apply() (and worse, a raw Python for-loop) calls a Python
  function once per row/element - Python function-call overhead dominates,
  and it's usually 10-100x slower on realistic data sizes.
- Reach for .apply() only when the operation truly can't be expressed as
  vectorized arithmetic/boolean logic/built-in Series methods.

In [ ]:
big = pd.DataFrame({"a": np.random.rand(200_000), "b": np.random.rand(200_000)})

start = time.perf_counter()
vectorized_result = big["a"] * big["b"]
vectorized_time = time.perf_counter() - start

start = time.perf_counter()
apply_result = big.apply(lambda row: row["a"] * row["b"], axis=1)
apply_time = time.perf_counter() - start

print(f"Same computation (a * b) over {len(big):,} rows:")
print(f"  vectorized (big['a'] * big['b'])      : {vectorized_time:.4f}s")
print(f"  .apply(axis=1) (Python call per row)  : {apply_time:.4f}s")
print(f"  .apply() was ~{apply_time / max(vectorized_time, 1e-9):.0f}x slower")
print("# Rule of thumb: if it's arithmetic, a comparison, or has a vectorized/Series method")
print("#   equivalent (.str, .dt, np.where, ...), do NOT reach for .apply() first.")

## 04. .sort_values() / .sort_index()

- .sort_values(by=col_or_list, ascending=True/False) sorts rows by value(s).
- .sort_index() sorts rows by their index label instead of any column.

In [ ]:
print("sales.sort_values('units', ascending=False):")
print(sales.sort_values("units", ascending=False))

print("\nsales.sort_values(['region', 'units']) (multi-column sort):")
print(sales.sort_values(["region", "units"]))

shuffled_index = sales.sample(frac=1, random_state=42)
print("\nshuffled_index.sort_index() (restores original row order by index label):")
print(shuffled_index.sort_index())

## 05. .value_counts(), .unique(), .nunique()

- .value_counts() -> counts of each distinct value, sorted descending by
  default - the fastest way to see a column's distribution.
- .unique() -> array of distinct values, in order of first appearance.
- .nunique() -> count of distinct values (a single number).

In [ ]:
print("sales['product'].value_counts():\n", sales["product"].value_counts(), sep="")
print("\nsales['product'].unique() =", sales["product"].unique())
print("sales['product'].nunique() =", sales["product"].nunique())
print("sales['product'].value_counts(normalize=True) (proportions instead of counts):")
print(sales["product"].value_counts(normalize=True))

## 06. .replace()

- .replace(old, new) swaps specific values for new ones - works with a
  single old->new pair, a dict of many pairs, or per-column dicts - unlike
  .map(), unmatched values are left UNCHANGED rather than becoming NaN.

In [ ]:
print("sales['region'].replace({'East': 'EAST', 'West': 'WEST'}):")
print(sales["region"].replace({"East": "EAST", "West": "WEST"}).tolist())
print("\n# Contrast with .map(): .replace() leaves values NOT in the dict untouched,")
print("#   .map() turns values NOT in the dict/Series into NaN.")

print("\n# End of Data Transformation")